# 🌾 AgriSmart — Satellite Images Preprocessing & Analysis


**What's Special:**
- ✅ All vegetation indices (NDVI, EVI, NDMI, NDWI, SAVI)
- ✅ K-Means clustering
- ✅ Complete interpretations

**Strategy:**
1. Load bands → Calculate index → Save to disk → Clear RAM → Repeat
2. When needed for visualization, load from disk temporarily
3. Use downsampling (2x) as safety margin



---
## 📦 Part 1: Setup

In [ ]:
!pip install rasterio matplotlib seaborn scikit-learn pandas numpy -q
print("✅ Packages installed")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import gc
import rasterio
from rasterio.enums import Resampling
from sklearn.cluster import MiniBatchKMeans

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Create temp directory for intermediate files
TEMP_DIR = Path('temp_indices')
TEMP_DIR.mkdir(exist_ok=True)

def clear_memory():
    """Aggressive memory cleanup"""
    gc.collect()
    
print("✅ Setup complete")

---
## 📂 Part 2: Load Data (Smart Loading)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Path to your GeoTIFF
tiff_path = '/content/drive/MyDrive/GEE_Exports/S2_Beja_2025.tif'

# Downsample factor (2 = safe, 1 = full resolution if you have enough RAM)
DOWNSAMPLE = 2

print("📊 Loading image metadata...")
with rasterio.open(tiff_path) as src:
    print("=" * 70)
    print(f"Original size: {src.width} x {src.height} pixels")
    print(f"Bands: {src.count}")
    print(f"CRS: {src.crs}")
    print(f"Resolution: {src.res[0]:.1f}m")
    
    # Calculate area
    area_km2 = (src.width * src.res[0] * src.height * src.res[1]) / 1e6
    print(f"Area: {area_km2:.2f} km²")
    
    # Store metadata
    meta = src.meta.copy()
    original_shape = (src.height, src.width)
    
    # Load with downsampling
    if DOWNSAMPLE > 1:
        print(f"\n🔧 Downsampling by {DOWNSAMPLE}x to save RAM...")
        data = src.read(
            out_shape=(src.count, src.height // DOWNSAMPLE, src.width // DOWNSAMPLE),
            resampling=Resampling.average
        )
    else:
        print("\n📥 Loading at full resolution...")
        data = src.read()
    
    meta['height'] = data.shape[1]
    meta['width'] = data.shape[2]

# Transpose to (H, W, C)
data = data.transpose(1, 2, 0)
h, w, c = data.shape

print(f"\n✅ Loaded: {w} x {h} pixels")
print(f"   RAM: {data.nbytes / 1e9:.2f} GB")
print("=" * 70)

# Extract bands
blue = data[:, :, 0]
green = data[:, :, 1]
red = data[:, :, 2]
nir = data[:, :, 3]

# Delete full array
del data
clear_memory()


---
## 🎨 Part 3: RGB Visualization

In [ ]:
# Create RGB (temp array, will be deleted)
print("Creating RGB composite...")
rgb = np.dstack([red, green, blue])
p2, p98 = np.percentile(rgb[rgb > 0], (2, 98))
rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(rgb)
axes[0].set_title('True Color RGB', fontsize=14, fontweight='bold')
axes[0].axis('off')

# False color
false_color = np.dstack([nir, red, green])
p2, p98 = np.percentile(false_color[false_color > 0], (2, 98))
false_color = np.clip((false_color - p2) / (p98 - p2), 0, 1)

axes[1].imshow(false_color)
axes[1].set_title('False Color (NIR-R-G)\nVegetation = Red', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('01_rgb_composites.png', dpi=150, bbox_inches='tight')
plt.show()

# Clean up
del rgb, false_color
clear_memory()


---
## 📊 Part 4: Calculate Indices ONE AT A TIME (RAM-Safe)

**Strategy:** Calculate → Save to disk → Clear RAM → Next index

In [ ]:
def safe_division(num, denom):
    """Safe division"""
    with np.errstate(divide='ignore', invalid='ignore'):
        result = np.divide(num, denom)
        result[np.isnan(result)] = 0
        result[np.isinf(result)] = 0
    return result

def save_index_to_disk(index_array, name):
    """Save index as numpy array"""
    np.save(TEMP_DIR / f'{name}.npy', index_array)
    print(f"   💾 Saved {name} to disk")

def load_index_from_disk(name):
    """Load index from disk"""
    return np.load(TEMP_DIR / f'{name}.npy')

print("🔄 Calculating indices (one at a time)...\n")

# 1. NDVI
print("[1/5] NDVI...")
ndvi = safe_division(nir - red, nir + red)
save_index_to_disk(ndvi, 'ndvi')
ndvi_stats = {'min': ndvi.min(), 'max': ndvi.max(), 'mean': ndvi.mean(), 'std': ndvi.std()}
print(f"   Stats: min={ndvi_stats['min']:.3f}, max={ndvi_stats['max']:.3f}, mean={ndvi_stats['mean']:.3f}")
del ndvi
clear_memory()

# 2. EVI
print("\n[2/5] EVI...")
evi = 2.5 * safe_division(nir - red, nir + 6*red - 7.5*blue + 1)
save_index_to_disk(evi, 'evi')
evi_stats = {'min': evi.min(), 'max': evi.max(), 'mean': evi.mean(), 'std': evi.std()}
print(f"   Stats: min={evi_stats['min']:.3f}, max={evi_stats['max']:.3f}, mean={evi_stats['mean']:.3f}")
del evi
clear_memory()

# 3. NDMI
print("\n[3/5] NDMI...")
ndmi = safe_division(nir - green, nir + green)
save_index_to_disk(ndmi, 'ndmi')
ndmi_stats = {'min': ndmi.min(), 'max': ndmi.max(), 'mean': ndmi.mean(), 'std': ndmi.std()}
print(f"   Stats: min={ndmi_stats['min']:.3f}, max={ndmi_stats['max']:.3f}, mean={ndmi_stats['mean']:.3f}")
del ndmi
clear_memory()

# 4. NDWI
print("\n[4/5] NDWI...")
ndwi = safe_division(green - nir, green + nir)
save_index_to_disk(ndwi, 'ndwi')
ndwi_stats = {'min': ndwi.min(), 'max': ndwi.max(), 'mean': ndwi.mean(), 'std': ndwi.std()}
print(f"   Stats: min={ndwi_stats['min']:.3f}, max={ndwi_stats['max']:.3f}, mean={ndwi_stats['mean']:.3f}")
del ndwi
clear_memory()

# 5. SAVI
print("\n[5/5] SAVI...")
L = 0.5
savi = safe_division((nir - red) * (1 + L), nir + red + L)
save_index_to_disk(savi, 'savi')
savi_stats = {'min': savi.min(), 'max': savi.max(), 'mean': savi.mean(), 'std': savi.std()}
print(f"   Stats: min={savi_stats['min']:.3f}, max={savi_stats['max']:.3f}, mean={savi_stats['mean']:.3f}")
del savi
clear_memory()


---
## 📊 Part 5: Visualize Indices (Load Temporarily)

In [ ]:
# Visualize all indices (load one at a time for plotting)
print("Creating visualization...")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# NDVI
ndvi = load_index_from_disk('ndvi')
im1 = axes[0, 0].imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[0, 0].set_title('NDVI\n(Vegetation Health)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')
plt.colorbar(im1, ax=axes[0, 0], fraction=0.046)
del ndvi

# EVI
evi = load_index_from_disk('evi')
im2 = axes[0, 1].imshow(evi, cmap='RdYlGn', vmin=-0.2, vmax=1.0)
axes[0, 1].set_title('EVI\n(Enhanced Vegetation)', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')
plt.colorbar(im2, ax=axes[0, 1], fraction=0.046)
del evi

# SAVI
savi = load_index_from_disk('savi')
im3 = axes[0, 2].imshow(savi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[0, 2].set_title('SAVI\n(Soil Adjusted)', fontsize=12, fontweight='bold')
axes[0, 2].axis('off')
plt.colorbar(im3, ax=axes[0, 2], fraction=0.046)
del savi

# NDMI
ndmi = load_index_from_disk('ndmi')
im4 = axes[1, 0].imshow(ndmi, cmap='Blues', vmin=-0.5, vmax=0.5)
axes[1, 0].set_title('NDMI\n(Moisture)', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')
plt.colorbar(im4, ax=axes[1, 0], fraction=0.046)
del ndmi

# NDWI
ndwi = load_index_from_disk('ndwi')
im5 = axes[1, 1].imshow(ndwi, cmap='Blues', vmin=-0.5, vmax=0.5)
axes[1, 1].set_title('NDWI\n(Water)', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')
plt.colorbar(im5, ax=axes[1, 1], fraction=0.046)
del ndwi

# RGB reference
rgb = np.dstack([red, green, blue])
p2, p98 = np.percentile(rgb[rgb > 0], (2, 98))
rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)
axes[1, 2].imshow(rgb)
axes[1, 2].set_title('RGB Reference', fontsize=12, fontweight='bold')
axes[1, 2].axis('off')
del rgb

plt.suptitle('Vegetation Indices - Béja Region', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('02_all_indices.png', dpi=150, bbox_inches='tight')
plt.show()

clear_memory()


---
## 📈 Part 6: NDVI Classification & Analysis

In [ ]:
# Load NDVI for classification
ndvi = load_index_from_disk('ndvi')

# Classification
classes = np.zeros_like(ndvi, dtype=np.uint8)
classes[ndvi < 0] = 0
classes[(ndvi >= 0) & (ndvi < 0.2)] = 1
classes[(ndvi >= 0.2) & (ndvi < 0.3)] = 2
classes[(ndvi >= 0.3) & (ndvi < 0.5)] = 3
classes[(ndvi >= 0.5) & (ndvi < 0.7)] = 4
classes[ndvi >= 0.7] = 5

# Save classification
save_index_to_disk(classes, 'ndvi_classes')

# Statistics
class_names = ['Water/Rock', 'Bare Soil', 'Sparse Veg', 'Moderate Veg', 'Dense Veg', 'Very Dense']
class_colors = ['#8B4513', '#D2B48C', '#FFFF99', '#90EE90', '#228B22', '#006400']

print("\n📊 NDVI CLASSIFICATION")
print("=" * 70)
coverage_data = []
for i in range(6):
    count = np.sum(classes == i)
    pct = (count / classes.size) * 100
    area_km2 = (count * 10 * 10) / 1e6
    coverage_data.append({'Class': class_names[i], 'Pixels': count, 'Percentage': pct, 'Area_km2': area_km2})
    print(f"{class_names[i]:15} | {count:10,} ({pct:5.2f}%) | {area_km2:6.2f} km²")

coverage_df = pd.DataFrame(coverage_data)
print("=" * 70)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# NDVI continuous
im1 = axes[0].imshow(ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[0].set_title('NDVI (Continuous)', fontsize=14, fontweight='bold')
axes[0].axis('off')
plt.colorbar(im1, ax=axes[0], fraction=0.046)

# Classified
from matplotlib.colors import ListedColormap
cmap = ListedColormap(class_colors)
im2 = axes[1].imshow(classes, cmap=cmap, vmin=0, vmax=5)
axes[1].set_title('NDVI Classification', fontsize=14, fontweight='bold')
axes[1].axis('off')

import matplotlib.patches as mpatches
patches = [mpatches.Patch(color=class_colors[i], label=class_names[i]) for i in range(6)]
axes[1].legend(handles=patches, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('03_ndvi_classification.png', dpi=150, bbox_inches='tight')
plt.show()

# Clean up but keep classes for clustering
del ndvi
clear_memory()


---
## 🎯 Part 7: K-Means Clustering (Memory-Efficient)

In [ ]:
print("🔄 Preparing features for clustering...")

# Load only needed indices
ndvi = load_index_from_disk('ndvi')
evi = load_index_from_disk('evi')

# Stack features (only 6 instead of all bands + indices)
features = np.dstack([red, green, blue, nir, ndvi, evi])
h_f, w_f, c_f = features.shape

# Normalize
features_norm = (features - np.nanmean(features, axis=(0, 1))) / (np.nanstd(features, axis=(0, 1)) + 1e-8)

# Reshape
pixels = features_norm.reshape(-1, c_f)
valid = ~np.any(np.isnan(pixels) | np.isinf(pixels), axis=1)
valid_pixels = pixels[valid]

del features, features_norm
clear_memory()

print(f"   Valid pixels: {valid_pixels.shape[0]:,}")

# MiniBatch K-Means
n_clusters = 6
print(f"\n🔄 Running MiniBatch K-Means ({n_clusters} clusters)...")

kmeans = MiniBatchKMeans(
    n_clusters=n_clusters,
    batch_size=10000,
    random_state=42,
    verbose=0
)
kmeans.fit(valid_pixels)

# Predict
labels = np.zeros(pixels.shape[0], dtype=np.uint8)
labels[valid] = kmeans.predict(valid_pixels)
clustered = labels.reshape(h_f, w_f)

# Save
save_index_to_disk(clustered, 'clusters')

# Analyze clusters
print("\n📊 CLUSTER ANALYSIS")
print("=" * 70)
cluster_data = []
for i in range(n_clusters):
    mask = clustered == i
    count = np.sum(mask)
    pct = (count / clustered.size) * 100
    avg_ndvi = np.mean(ndvi[mask]) if count > 0 else 0
    avg_evi = np.mean(evi[mask]) if count > 0 else 0
    
    cluster_data.append({
        'Cluster': i, 'Pixels': count, 'Percentage': pct,
        'Avg_NDVI': avg_ndvi, 'Avg_EVI': avg_evi
    })
    print(f"Cluster {i} | {count:10,} ({pct:5.2f}%) | NDVI={avg_ndvi:.3f} | EVI={avg_evi:.3f}")

cluster_df = pd.DataFrame(cluster_data)
print("=" * 70)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

rgb = np.dstack([red, green, blue])
p2, p98 = np.percentile(rgb[rgb > 0], (2, 98))
rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)

axes[0].imshow(rgb)
axes[0].set_title('RGB', fontsize=14, fontweight='bold')
axes[0].axis('off')

im = axes[1].imshow(clustered, cmap='tab10')
axes[1].set_title(f'K-Means Clustering ({n_clusters} classes)', fontsize=14, fontweight='bold')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.savefig('04_kmeans_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

# Clean up
del ndvi, evi, rgb, pixels, valid_pixels, labels, clustered
clear_memory()


---
## 📊 Part 8: Statistical Analysis

In [ ]:
# Create comprehensive statistics table
stats_df = pd.DataFrame({
    'Index': ['NDVI', 'EVI', 'NDMI', 'NDWI', 'SAVI'],
    'Min': [ndvi_stats['min'], evi_stats['min'], ndmi_stats['min'], ndwi_stats['min'], savi_stats['min']],
    'Max': [ndvi_stats['max'], evi_stats['max'], ndmi_stats['max'], ndwi_stats['max'], savi_stats['max']],
    'Mean': [ndvi_stats['mean'], evi_stats['mean'], ndmi_stats['mean'], ndwi_stats['mean'], savi_stats['mean']],
    'Std': [ndvi_stats['std'], evi_stats['std'], ndmi_stats['std'], ndwi_stats['std'], savi_stats['std']]
})

print("\n📊 VEGETATION INDEX STATISTICS")
print("=" * 70)
print(stats_df.to_string(index=False))
print("=" * 70)

# Distribution plots (load temporarily)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

indices = ['ndvi', 'evi', 'ndmi', 'ndwi', 'savi']
titles = ['NDVI', 'EVI', 'NDMI', 'NDWI', 'SAVI']
colors_hist = ['green', 'darkgreen', 'blue', 'cyan', 'olive']

for i, (idx_name, title, color) in enumerate(zip(indices, titles, colors_hist)):
    row = i // 3
    col = i % 3
    
    data = load_index_from_disk(idx_name)
    axes[row, col].hist(data.flatten(), bins=50, color=color, alpha=0.7, edgecolor='black')
    axes[row, col].axvline(data.mean(), color='red', linestyle='--', linewidth=2, 
                          label=f'Mean: {data.mean():.3f}')
    axes[row, col].set_xlabel(title, fontsize=11)
    axes[row, col].set_ylabel('Frequency', fontsize=11)
    axes[row, col].set_title(f'{title} Distribution', fontsize=12, fontweight='bold')
    axes[row, col].legend()
    axes[row, col].grid(alpha=0.3)
    del data

# Cluster distribution
clusters = load_index_from_disk('clusters')
cluster_counts = [np.sum(clusters == i) for i in range(n_clusters)]
axes[1, 2].bar(range(n_clusters), cluster_counts, 
              color=plt.cm.tab10(np.linspace(0, 1, n_clusters)), edgecolor='black', linewidth=1.5)
axes[1, 2].set_xlabel('Cluster ID', fontsize=11)
axes[1, 2].set_ylabel('Pixel Count', fontsize=11)
axes[1, 2].set_title('Cluster Distribution', fontsize=12, fontweight='bold')
axes[1, 2].grid(axis='y', alpha=0.3)
del clusters

plt.suptitle('Statistical Distributions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('05_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

clear_memory()


---
## 💾 Part 9: Export All Results

In [ ]:
from pathlib import Path
output_dir = Path('agrismart_outputs')
output_dir.mkdir(exist_ok=True)

print("📦 Exporting results...\n")

# Export each index as GeoTIFF
index_names = ['ndvi', 'evi', 'ndmi', 'ndwi', 'savi', 'ndvi_classes', 'clusters']

for name in index_names:
    data = load_index_from_disk(name)
    
    meta_copy = meta.copy()
    meta_copy.update({
        'count': 1,
        'dtype': 'float32' if name not in ['ndvi_classes', 'clusters'] else 'uint8'
    })
    
    output_path = output_dir / f'{name}_beja.tif'
    with rasterio.open(output_path, 'w', **meta_copy) as dst:
        dst.write(data.astype(meta_copy['dtype']), 1)
    
    print(f"   ✓ {name.upper()}: {output_path}")
    del data

clear_memory()

# Export statistics
stats_df.to_csv(output_dir / 'index_statistics.csv', index=False)
coverage_df.to_csv(output_dir / 'ndvi_coverage.csv', index=False)
cluster_df.to_csv(output_dir / 'cluster_statistics.csv', index=False)

print(f"   ✓ CSV files saved")

# Summary report
report = f"""
{'='*70}
AGRISMART - COMPLETE ANALYSIS REPORT
Béja Region, Tunisia
{'='*70}

IMAGE INFO:
- Size: {w} x {h} pixels
- Area: {area_km2:.2f} km²
- Downsampling: {DOWNSAMPLE}x

VEGETATION INDICES:
{stats_df.to_string(index=False)}

NDVI CLASSIFICATION:
{coverage_df.to_string(index=False)}

K-MEANS CLUSTERING:
{cluster_df.to_string(index=False)}

OUTPUTS GENERATED:
1. 01_rgb_composites.png
2. 02_all_indices.png
3. 03_ndvi_classification.png
4. 04_kmeans_clustering.png
5. 05_distributions.png
6-12. GeoTIFF files for all indices
13-15. CSV statistics files
"""

with open(output_dir / 'analysis_report.txt', 'w') as f:
    f.write(report)

print(f"   ✓ Report saved\n")
print(f"✅ All outputs in: {output_dir.absolute()}")
print(report)

---
## 🧹 Part 10: Final Cleanup

In [ ]:
# Clean up bands from memory
del blue, green, red, nir, classes
clear_memory()

print("🧹 Final cleanup complete!")

---
## 📖 Part 11: Interpretation Guide

### 🔍 Index Meanings:

**NDVI (Normalized Difference Vegetation Index)**
- Range: -1 to +1
- < 0: Water, wet soil
- 0-0.2: Bare soil, rock
- 0.2-0.5: Sparse to moderate vegetation
- 0.5-0.8: Dense healthy vegetation
- > 0.8: Very dense vegetation

**EVI (Enhanced Vegetation Index)**
- Better for high biomass areas
- Less atmospheric interference
- Similar interpretation to NDVI

**NDMI (Moisture Index)**
- < 0: Water stress, dry vegetation
- 0-0.3: Moderate moisture
- > 0.3: High moisture content

**NDWI (Water Index)**
- Highlights water bodies
- High values = water presence

**SAVI (Soil Adjusted)**
- Better for areas with exposed soil
- Reduces soil brightness effects

### 📊 For Your Report:

Include:
1. All 5 PNG visualizations
2. Statistics table (from CSV)
3. Land cover percentages
4. Cluster analysis
5. Interpretation of results

### 🚀 Next Steps:

1. Use GeoTIFFs in QGIS for spatial analysis
2. Build ML models using these indices as features
3. Temporal analysis (compare multiple dates)
4. Ground truth validation

**Memory Strategy Used:**
- ✅ Calculate indices one at a time
- ✅ Save to disk immediately
- ✅ Load only when needed
- ✅ Delete after use